# CMVA-GS Baseline Replication — Colab (GPU)
Runs the GPU-heavy steps (5 PPO Moral Agents + Aggregator) on a free T4.
**Runtime → Change runtime type → GPU** before running.

In [ ]:
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# Option A: clone your repo (after you push it to GitHub)
# !git clone https://github.com/<you>/project_nlp.git && %cd project_nlp
# Option B: upload src/ and scripts/ via the Files panel, then continue.
%cd /content

In [ ]:
!pip -q install transformers==5.0.0 trl==0.11.4 peft==0.22.0 datasets accelerate evaluate rouge-score scikit-learn beautifulsoup4 sentencepiece

In [ ]:
# Download + preprocess Social Chemistry 101
import os
os.makedirs('data/raw', exist_ok=True)
!wget -q -O data/raw/sc.zip https://storage.googleapis.com/ai2-mosaic-public/projects/social-chemistry/data/social-chem-101.zip
!cd data/raw && unzip -o -q sc.zip
!python3 src/prepare_data.py --raw data/raw/social-chem-101/social-chem-101.v1.0.tsv

In [ ]:
# Reward models (fast on GPU)
!python3 src/train_reward_models.py --max_train 12000 --epochs 2

In [ ]:
# PT (base) conformance
!python3 src/eval_conformance.py --label PT --n_prompts 300

In [ ]:
# Train the 5 Moral Agents (PPO). On T4 raise --steps for better separation.
for v in ['care','fairness','authority','sanctity','loyalty']:
    !python3 src/train_ppo_agent.py --value $v --steps 100 --n_prompts 1600

In [ ]:
# Agent conformance + Table 1
for v in ['care','fairness','authority','sanctity','loyalty']:
    !python3 src/eval_conformance.py --adapter models/agents/$v --label agent_$v --n_prompts 300
!python3 src/build_table.py

In [ ]:
# Aggregator + ROUGE (Figure 2)
!python3 src/train_aggregator.py --n_train 600
!python3 src/eval_rouge.py --n 200